In [0]:
import requests
import json
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import (
    StructType, StructField, StringType, 
    FloatType, IntegerType, TimestampType
)

# Storage config
storage_account = "theportfoliostorage"
container = "datalake"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", 
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", 
               dbutils.secrets.get(scope="de-portfolio-scope", key="sp-client-id"))
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", 
               dbutils.secrets.get(scope="de-portfolio-scope", key="sp-client-secret"))
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", 
               f"https://login.microsoftonline.com/{dbutils.secrets.get(scope='de-portfolio-scope', key='sp-tenant-id')}/oauth2/token")

print("Storage configured successfully")

In [0]:
# Hit the Carbon Intensity API for all UK regions
url = "https://api.carbonintensity.org.uk/regional"

response = requests.get(url, headers={"Accept": "application/json"})

print(f"Status code: {response.status_code}")
print(f"\nRaw response structure:")
print(json.dumps(response.json(), indent=2))

In [0]:
import time
from pyspark.sql import Row

def parse_carbon_response(response_json):
    data = response_json["data"][0]
    period_from = data["from"]
    period_to = data["to"]
    batch_id = int(time.time())
    
    intensity_rows = []
    generation_rows = []
    
    for region in data["regions"]:
        region_id = region["regionid"]
        short_name = region["shortname"]
        dno_region = region["dnoregion"]
        intensity_forecast = region["intensity"]["forecast"]
        intensity_index = region["intensity"]["index"]
        
        # One row per region for intensity table
        intensity_rows.append(Row(
            region_id=region_id,
            short_name=short_name,
            dno_region=dno_region,
            intensity_forecast=float(intensity_forecast),
            intensity_index=intensity_index,
            period_from=period_from,
            period_to=period_to,
            batch_id=batch_id
        ))
        
        # One row per fuel type for generation mix table
        for fuel in region["generationmix"]:
            generation_rows.append(Row(
                region_id=region_id,
                short_name=short_name,
                fuel_type=fuel["fuel"],
                percentage=float(fuel["perc"]),
                period_from=period_from,
                period_to=period_to,
                batch_id=batch_id
            ))
    
    return intensity_rows, generation_rows

# Parse the response we already have
intensity_rows, generation_rows = parse_carbon_response(response.json())

# Create DataFrames
df_intensity = spark.createDataFrame(intensity_rows) \
    .withColumn("_ingested_at", current_timestamp())

df_generation = spark.createDataFrame(generation_rows) \
    .withColumn("_ingested_at", current_timestamp())

print(f"Intensity rows: {df_intensity.count()}")
print(f"Generation mix rows: {df_generation.count()}")

print("\nIntensity sample:")
df_intensity.select(
    "region_id", "short_name", 
    "intensity_forecast", "intensity_index",
    "period_from"
).show(5, truncate=False)

print("\nGeneration mix sample:")
df_generation.select(
    "region_id", "short_name",
    "fuel_type", "percentage",
    "period_from"
).show(5, truncate=False)

In [0]:
# Write intensity table
df_intensity.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("energy.bronze.carbon_intensity")

print("Carbon intensity table written")

# Write generation mix table
df_generation.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("energy.bronze.generation_mix")

print("Generation mix table written")

# Verify
print("\nCarbon intensity row count:")
spark.sql("SELECT COUNT(*) as total FROM energy.bronze.carbon_intensity").show()

print("Generation mix row count:")
spark.sql("SELECT COUNT(*) as total FROM energy.bronze.generation_mix").show()

In [0]:
import time

def fetch_and_store_carbon_data():
    url = "https://api.carbonintensity.org.uk/regional"
    response = requests.get(url, headers={"Accept": "application/json"})
    
    if response.status_code != 200:
        raise Exception(f"API call failed with status {response.status_code}")
    
    intensity_rows, generation_rows = parse_carbon_response(response.json())
    
    df_intensity = spark.createDataFrame(intensity_rows) \
        .withColumn("_ingested_at", current_timestamp())
    
    df_generation = spark.createDataFrame(generation_rows) \
        .withColumn("_ingested_at", current_timestamp())
    
    df_intensity.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("energy.bronze.carbon_intensity")
    
    df_generation.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("energy.bronze.generation_mix")
    
    total_intensity = spark.sql(
        "SELECT COUNT(*) as total FROM energy.bronze.carbon_intensity"
    ).collect()[0][0]
    
    print(f"Batch complete - Total intensity rows: {total_intensity}")

# Run once per scheduled trigger
fetch_and_store_carbon_data()